In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.utils.parametrizations as parametrizations
import pandas as pd
import numpy as np
import joblib
import pymysql
from fastapi import FastAPI, HTTPException
from sklearn.preprocessing import MinMaxScaler
import traceback
import uvicorn

# ==========================================
# 1. KONFIGURÁCIÓ ÉS ADATBÁZIS BEÁLLÍTÁSOK
# ==========================================
load_dotenv()
DB_HOST = os.getenv("DB_HOST")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
DB_TABLE = "weather_live"

MODEL_PATH = "tcn_model.pth"
SCALER_PATH = "scaler.pkl"

INPUT_WINDOW = 168
OUTPUT_HORIZON = 48

FEATURES = [
    "temp_c", "dewpoint_c", "rh", "wind_speed", 
    "sp_hpa", "tp", "skt_c", "ssrd_wm2", "fg10_kmh", 
    "hour_sin", "hour_cos", "year_sin", "year_cos", "wind_dir_sin", "wind_dir_cos"
]
TARGETS = ["temp_c", "dewpoint_c", "rh", "wind_speed", "sp_hpa", "tp", "skt_c", "ssrd_wm2", "fg10_kmh", "wind_dir_sin", "wind_dir_cos"]
#TARGETS = ["temp_c", "rh", "ssrd_wm2"]

# ==========================================
# 2. MODEL OSZTÁLYOK (A betöltéshez szükségesek)
# ==========================================
class GroupScaler:
    # (A te GroupScaler osztályod, ahogy megírtad)
    def __init__(self, groups=None):
        self.groups = groups if groups else {}
        self.group_scalers = {}
        self.standalone_scaler = MinMaxScaler()
        self.all_features = []
        self.standalone_features = []
    
    def transform(self, df):
        df_out = df.copy()
        for name, cols in self.groups.items():
            active_cols = [c for c in cols if c in self.all_features]
            if active_cols:
                scaler = self.group_scalers[name]
                for col in active_cols:
                    df_out[col] = scaler.transform(df_out[[col]].values)
        if self.standalone_features:
            df_out[self.standalone_features] = self.standalone_scaler.transform(df_out[self.standalone_features])
        return df_out

    def inverse_transform_single_col(self, data_array, col_name):
        for name, cols in self.groups.items():
            if col_name in cols:
                scaler = self.group_scalers[name]
                return scaler.inverse_transform(data_array.reshape(-1, 1)).reshape(data_array.shape)
        if col_name in self.standalone_features:
            idx = self.standalone_features.index(col_name)
            dummy = np.zeros((len(data_array.flatten()), len(self.standalone_features)))
            dummy[:, idx] = data_array.flatten()
            inv_dummy = self.standalone_scaler.inverse_transform(dummy)
            return inv_dummy[:, idx].reshape(data_array.shape)
        return data_array

# Deffiniálom az azonos mértékegységű csoportokat.
SCALING_GROUPS = {
    "celsius": ["temp_c", "dewpoint_c", "skt_c"],
    "wind": ["wind_speed", "fg10_kmh"]
}

class ManualGroupScaler:
    def __init__(self, feature_limits):
        """
        feature_limits: dict, pl. {"temp_c": (-30.0, 45.0), "rh": (0.0, 100.0)}
        """
        self.limits = feature_limits
        self.all_features = []

    def fit(self, df, features):
        # Erre a függvényre a kompatibilitás miatt van szükség, de nem tanul semmit az adatokból!
        self.all_features = features
        return self

    def transform(self, df):
        df_out = df.copy()
        for col in self.all_features:
            if col in self.limits:
                f_min, f_max = self.limits[col]
                # Sima Min-Max skálázás képlettel: (x - min) / (max - min)
                df_out[col] = (df_out[col] - f_min) / (f_max - f_min)
            # Aminek nincs határa (pl. sin/cos oszlopok), azt békén hagyja, mert az eleve -1 és 1 között van
        return df_out

    def inverse_transform_single_col(self, data_array, col_name):
        """Visszaskálázás a valós fizikai dimenzióba"""
        if col_name in self.limits:
            f_min, f_max = self.limits[col_name]
            # Inverz képlet: x * (max - min) + min
            return data_array * (f_max - f_min) + f_min
        return data_array

# Határozzuk meg a fizikai korlátokat a te régiódra szabva
WEATHER_LIMITS = {
    "temp_c": (-100.0, 100.0),
    "dewpoint_c": (-100.0, +100.0),
    "skt_c": (-100.0, 100.0),       # Talajfelszín hőmérséklet kicsit tágabb
    "rh": (0.0, 100.0),
    "wind_speed": (0.0, 600.0),    # km/h vagy m/s szerint (igazítsd a saját adatodhoz!)
    "fg10_kmh": (0.0, 600.0),
    "wind_dir": (0.0, 360.0),
    "sp_hpa": (800.0, 1100.0),     # Légnyomás hPa
    "tp": (0.0, 400.0),             # Csapadék max mm
    "ssrd_wm2": (0.0, 2000.0)      # Globálsugárzás max
}

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(ResidualBlock, self).__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = parametrizations.weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.conv2 = parametrizations.weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.conv2, self.chomp2, self.relu2)
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, num_inputs, output_horizon, num_targets, num_channels=[64, 64, 128], kernel_size=3):
        super(TCNModel, self).__init__()
        self.num_targets = num_targets
        self.output_horizon = output_horizon
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(ResidualBlock(in_channels, out_channels, kernel_size, dilation=dilation_size))
        self.tcn = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_horizon * num_targets)

    def forward(self, x):
        x = x.transpose(1, 2)
        y = self.tcn(x)
        out = self.linear(y[:, :, -1])
        out = out.view(-1, self.output_horizon, self.num_targets)
        return out

# ==========================================
# 3. SZERVER ÉS MODELL INICIALIZÁLÁSA
# ==========================================
app = FastAPI(title="Weather TCN Prediction API", version="1.0")

print("Modellek és Scaler betöltése a memóriába...")
device = torch.device("cpu") # Az API-nak bőven elég a CPU a gyors predikcióhoz

# Skálázó betöltése
try:
    scaler = joblib.load(SCALER_PATH)
except Exception as e:
    print(f"Hiba a scaler betöltésekor: {e}")
    exit(1)

# Modell betöltése
model = TCNModel(num_inputs=len(FEATURES), output_horizon=OUTPUT_HORIZON, num_targets=len(TARGETS))
try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval() # Fontos: kiértékelő mód
except Exception as e:
    print(f"Hiba a modell betöltésekor: {e}")
    exit(1)
print("Rendszer készen áll!")

def encode_cyclic_time(df, col):
    # Óra kódolása (0-23, tehát 24 egység!)
    # Megjegyzés: itt érdemesebb 24-gyel osztani, mert 0-tól 23-ig 24 állapot van
    #LINEÁRIS IDŐ: Unix timestamp (másodpercek), ez azért fontos, hogy az évek közötti változást is tudja értelmezni majd a model.
    df = df.sort_values(col).reset_index(drop=True)
    df['time_linear'] = df[col].astype('int64') // 10**9

    #Napok tördelése, sin és cos ciklusba tördelem így reprezentál egy napot.
    hour = df[col].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Év napja (1-365 vagy 1-366)
    day_of_year = df[col].dt.dayofyear
    
    # Megnézzük, szökőév-e az adott sor dátuma
    # Ha szökőév, 366-tal osztunk, ha nem, 365-tel
    days_in_year = np.where(df[col].dt.is_leap_year, 366, 365)

    #Hasonlóan a napokhoz az éveket is körkörös sin és cos ciklusokba tördelem
    df['year_sin'] = np.sin(2 * np.pi * (day_of_year - 1) / days_in_year)
    df['year_cos'] = np.cos(2 * np.pi * (day_of_year - 1) / days_in_year)

     # A kódod elején az encode_cyclic_time mintájára:
    df['wind_dir_sin'] = np.sin(2 * np.pi * df['wind_dir'] / 360.0)
    df['wind_dir_cos'] = np.cos(2 * np.pi * df['wind_dir'] / 360.0)
    
    return df

def calculate_condition(row):
    """
    Kiszámítja az időjárási állapotot (condition) a fizikai paraméterek alapján.
    """
    temp = row["temp_c"]
    rh = row["rh"]
    
    # --- JAVÍTÁS: Átváltjuk mm-re a könnyebb logikai vizsgálathoz! ---
    tp_mm = row["tp"] * 1000.0 
    
    ws = row["wind_speed"]
    fg = row["fg10_kmh"]
    ssrd = row["ssrd_wm2"]
    
    # 1. CSAPADÉK LOGIKA
    # 0.1 mm már egy mérhető, valós csapadék (kiszűri a 0.00001-es modell zajt)
    if tp_mm > 0.1: 
        if temp <= 0:
            return "snowy"
        elif 0 < temp < 2:
            return "snowy-rainy"
        
        # 5.0 mm / óra már határozottan heves esőzésnek (zápornak) számít
        if tp_mm > 5.0: 
            return "pouring"
        
        if fg > 50: # Viharos szél + eső
            return "lightning-rainy"
        
        return "rainy"

    # 2. LÁTÁSVÁSZONYOK (Köd)
    # Harmatpont és hőmérséklet különbsége alapján (ha közel van, ködös)
    if abs(temp - row["dewpoint_c"]) < 1.0 and ws < 15:
        return "fog"

    # 3. SZÉL
    is_windy = ws > 25 or fg > 45

    # 4. NAPSUGÁRZÁS / FELHŐZET (Nappal vagy Éjszaka)
    is_daylight = ssrd > 10 # Ha van mérhető sugárzás

    if is_daylight:
        # Napsütés vs Felhőzet
        if ssrd > 400:
            return "sunny"
        elif 100 < ssrd <= 400:
            return "partlycloudy"
        else:
            return "windy-variant" if is_windy else "cloudy"
    else:
        # Éjszakai logika
        if rh > 85:
            return "cloudy"
        return "clear-night"

    if is_windy:
        return "windy"

    return "exceptional"

# ==========================================
# 4. API VÉGPONT (Endpoint)
# ==========================================
@app.get("/predict")
def get_weather_prediction():
    try:
        # 1. Adatbázis kapcsolat
        conn = pymysql.connect(host=DB_HOST, user=DB_USER, password=DB_PASS, database=DB_NAME)
        
        # 2. Utolsó 168 óra lekérdezése (ID vagy időbélyeg alapján csökkenő, majd megfordítva)
        # Feltételezzük, hogy van egy 'valid_time' vagy 'id' oszlop a sorrendhez
        query = f"SELECT valid_time, temp_c, dewpoint_c, rh, wind_speed, wind_dir, sp_hpa, tp, skt_c, ssrd_wm2, fg10_kmh FROM {DB_TABLE} ORDER BY valid_time DESC LIMIT {INPUT_WINDOW}"
        df = pd.read_sql(query, conn)
        conn.close()

        df = encode_cyclic_time(df, "valid_time")

        df_local = df.copy() #Bemásolom a globális változóból az adattömböt
        
        # ---------------------------------------------------------
        # 5. SKÁLÁZÁS (JAVÍTVA)
        # ---------------------------------------------------------
        # Nem hozunk létre új Scalert, és NEM hívunk .fit()-et!
        # A globális, memóriába betöltött `scaler` objektumot használjuk.
        
       # A betöltött ManualGroupScaler példányt közvetlenül használjuk a FEATURES listával
        scaler.all_features = FEATURES
        df_scaled = scaler.transform(df_local)
        
        # --- Adattisztítás ---
        df_scaled = df_scaled.ffill().bfill()
        
        # Konvertálás float32-re a PyTorch-hoz (Nagybetűs FEATURES!)
        data_array = df_scaled[FEATURES].values.astype(np.float32)
        """
        # --- DEBUG TESZT ---
        print("\n=== BEMENETI ADATOK SKÁLÁZOTT ÉRTÉKEI ===")
        for col in df_scaled.columns:
            print(f"{col}: min={df_scaled[col].min():.2f}, max={df_scaled[col].max():.2f}")
        print("=========================================\n")
        """
        # 6. Tenzorrá alakítás: [Batch(1), Time(168), Features(15)]
        tensor_x = torch.tensor(data_array).unsqueeze(0).to(device)

        # 7. Predikció futtatása
        with torch.no_grad():
            output = model(tensor_x) # Forma: [1, 48, 10]
        
        # Visszaalakítás numpy tömbbé: [48, 10]
        pred_array = output.squeeze(0).numpy()

        # ---------------------------------------------------------
        # 8. VISSZASKÁLÁZÁS (JAVÍTVA)
        # ---------------------------------------------------------
        pred_real = np.zeros_like(pred_array)
        for t, col_name in enumerate(TARGETS):
            # 2D-be kényszerítjük (48, 1) és levágjuk az extrém kilengéseket
            col_data = pred_array[:, t].reshape(-1, 1)
            col_data = np.clip(col_data, -0.2, 1.2)
            
            # Visszaskálázás
            res = scaler.inverse_transform_single_col(col_data, col_name)
            pred_real[:, t] = res.flatten()

        # 9. JSON válasz összeállítása
        predictions_list = []
        for h in range(OUTPUT_HORIZON):
            # Kivesszük a visszaskálázott értékeket a TARGETS lista aktuális indexei alapján
            p_temp       = float(pred_real[h, TARGETS.index("temp_c")])
            p_dewpoint   = float(pred_real[h, TARGETS.index("dewpoint_c")])
            p_rh         = float(pred_real[h, TARGETS.index("rh")])
            p_wind_speed = float(pred_real[h, TARGETS.index("wind_speed")])
            p_sp_hpa     = float(pred_real[h, TARGETS.index("sp_hpa")])
            p_tp         = float(pred_real[h, TARGETS.index("tp")])
            p_skt_c      = float(pred_real[h, TARGETS.index("skt_c")])
            p_ssrd_wm2   = float(pred_real[h, TARGETS.index("ssrd_wm2")])
            p_fg10_kmh   = float(pred_real[h, TARGETS.index("fg10_kmh")])
            
            # --- SZÉLIRÁNY REKONSTRUKCIÓ ---
            # Kiszámoljuk a sin és cos komponensekből a valós fokot
            w_sin = float(pred_real[h, TARGETS.index("wind_dir_sin")])
            w_cos = float(pred_real[h, TARGETS.index("wind_dir_cos")])
            w_dir_deg = np.degrees(np.arctan2(w_sin, w_cos))
            w_dir_deg = int(round(w_dir_deg % 360, 0)) # Biztosítjuk a 0-360 fokos sávot

            # Ideiglenes szótár az időjárási állapot (condition) kiszámításához
            p_raw = {
                "temp_c": p_temp, "dewpoint_c": p_dewpoint, "rh": p_rh, 
                "wind_speed": p_wind_speed, "wind_dir": float(w_dir_deg), 
                "sp_hpa": p_sp_hpa, "tp": p_tp, "skt_c": p_skt_c, 
                "ssrd_wm2": p_ssrd_wm2, "fg10_kmh": p_fg10_kmh
            }

            current_condition = calculate_condition(p_raw)
            
            predictions_list.append({
                "hour_offset": h + 1,
                "temp_c":      round(p_raw["temp_c"], 2),
                "dewpoint_c":  round(p_raw["dewpoint_c"], 2),
                "rh":          max(0.0, min(100.0, round(p_raw["rh"], 2))),
                "wind_speed":  max(0.0, round(p_raw["wind_speed"], 2)),
                "wind_dir":    int(round(p_raw["wind_dir"] % 360, 0)),
                "sp_hpa":      round(p_raw["sp_hpa"], 1),
                "tp":          max(0.0, round(p_raw["tp"] * 1000, 4)),
                "skt_c":       round(p_raw["skt_c"], 2),
                "ssrd_wm2":    max(0.0, round(p_raw["ssrd_wm2"], 1)),
                "fg10_kmh":    max(0.0, round(p_raw["fg10_kmh"], 2)),
                "condition":   current_condition
            })
            
        return {
            "status": "success",
            "horizon": OUTPUT_HORIZON,
            "predictions": predictions_list
        }

    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


# ==========================================
# 5. SZERVER INDÍTÁSA JUPYTERBEN
# ==========================================
if __name__ == "__main__":
    print("Szerver indítása a Jupyter Notebook aszinkron motorjával...")
    
    # Közvetlenül a Jupyter futó eseményhurkát (event loop) használjuk:
    config = uvicorn.Config(app, host="0.0.0.0", port=8055)
    server = uvicorn.Server(config)
    
    # Itt a varázslat: a beépített 'await' parancsot használjuk!
    await server.serve()
"""
# ==========================================
# 5. SZERVER INDÍTÁSA
# ==========================================
if __name__ == "__main__":
    import uvicorn
    print("Szerver indítása...")
    # Ez a metódus felismeri a futtatási környezetet és elindítja a szervert
    uvicorn.run(app, host="0.0.0.0", port=8055)
    """

INFO:     Started server process [407414]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8055 (Press CTRL+C to quit)


Modellek és Scaler betöltése a memóriába...
Rendszer készen áll!
Szerver indítása a Jupyter Notebook aszinkron motorjával...
INFO:     192.168.21.105:40353 - "GET /docs HTTP/1.1" 200 OK
INFO:     192.168.21.105:40353 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     192.168.21.105:54191 - "GET /predict HTTP/1.1" 200 OK


/tmp/ipykernel_407414/3334151104.py:300: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


INFO:     192.168.21.95:44378 - "GET /predict HTTP/1.1" 200 OK


/tmp/ipykernel_407414/3334151104.py:300: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [407414]


'\n# ==========================================\n# 5. SZERVER INDÍTÁSA\n# ==========================================\nif __name__ == "__main__":\n    import uvicorn\n    print("Szerver indítása...")\n    # Ez a metódus felismeri a futtatási környezetet és elindítja a szervert\n    uvicorn.run(app, host="0.0.0.0", port=8055)\n    '